# conv-windowing-2d — worked example 1: Full 2-D conv from a window view plus einsum

> Worked example from [Delta Drills](https://delta-drills.vercel.app). Atom: `conv-windowing-2d`.

**This is a worked example — read it, run it, follow the reasoning.** It is study material, not a graded drill (no completion beacon). When the steps feel obvious, move to the faded version, then the full drill.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
import matplotlib.pyplot as plt

## Concept

A stride-1 2-D convolution is just a windowed contraction. Build the `(B, IC, OH, OW, KH, KW)` view with `as_strided` (middle `(s_h, s_w)` steps between windows, trailing `(s_h, s_w)` steps within a window), then contract it against a kernel `(OC, IC, KH, KW)` with `einsum`. The result `(B, OC, OH, OW)` must equal `F.conv2d`.

## Worked solution

**Step 1 — output shape.** For stride-1, `OH = H - KH + 1` and `OW = W - KW + 1`. Each output position is one valid placement of the kernel.

**Step 2 — read the input strides.** `x.stride()` gives `(s_b, s_ic, s_h, s_w)` in *element* units. We reuse these to address overlapping windows without copying.

**Step 3 — build the view.** Call `as_strided(size=(B, IC, OH, OW, KH, KW), stride=(s_b, s_ic, s_h, s_w, s_h, s_w))`. The middle `(s_h, s_w)` advances *between* windows (1 input pixel per output step); the trailing `(s_h, s_w)` advances *within* a window. Adjacent windows overlap by `KH-1` rows / `KW-1` cols — that overlap is exactly why the view shares storage rather than copying.

**Step 4 — contract.** `einsum('b ic oh ow kh kw, oc ic kh kw -> b oc oh ow')` sums the elementwise product of each window with each output filter over `(ic, kh, kw)`. That is the definition of cross-correlation, which is what `F.conv2d` computes.

**Step 5 — verify.** Compare against `F.conv2d(x, w)` (no bias, no padding) with `torch.allclose`. They match because both reduce the same `(ic, kh, kw)` axes over the same windows.

In [ ]:
import torch.nn.functional as F
from einops import einsum

def conv2d_via_windows(x: Tensor, w: Tensor) -> Tensor:
    B, IC, H, W = x.shape
    OC, IC2, KH, KW = w.shape
    assert IC == IC2
    OH = H - KH + 1
    OW = W - KW + 1
    s_b, s_ic, s_h, s_w = x.stride()
    windows = x.as_strided(
        size=(B, IC, OH, OW, KH, KW),
        stride=(s_b, s_ic, s_h, s_w, s_h, s_w),
    )
    return einsum(windows, w, 'b ic oh ow kh kw, oc ic kh kw -> b oc oh ow')

t.manual_seed(0)
x = t.randn(2, 3, 7, 8)
w = t.randn(4, 3, 3, 3)
out = conv2d_via_windows(x, w)
ref = F.conv2d(x, w)
print(out.shape, bool(t.allclose(out, ref, atol=1e-4)))